# Scenario Playground Step 11: S&P 500 XGBoost Prototype

Objective: determine whether a small, shallow, regularized XGBoost candidate adds honest S&P 500 validation value beyond the fixed Step 10 Ridge and naive baselines.

Scope: event-aware sampling, the approved 584-row/58-feature matrix, frozen development/validation roles, and the target-end-date purge remain unchanged. Only S&P 500 is trained. The reader stops before all 114 locked final-test rows.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "scenario_modeling").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Run this notebook from inside the InsightPulse repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scenario_modeling.src.xgboost_prototype import (
    build_xgboost_prototype_artifacts,
    persist_xgboost_prototype_artifacts,
)

artifacts = build_xgboost_prototype_artifacts()
output_paths = persist_xgboost_prototype_artifacts(artifacts)
summary = artifacts["summary"]
summary["training_gate"], output_paths

## Frozen data and lock boundary

The same bounded loader used by Step 10 parses 470 pre-final-test rows, uses 354 retained development rows for fitting, predicts 110 retained validation rows, excludes six boundary-purge rows, and parses no locked target row.

In [ ]:
guard = pd.Series(summary["matrix_guard"], name="value")
display(guard.to_frame())
assert summary["sampling_decision"].startswith("EVENT_AWARE")
assert summary["feature_count"] == 58
assert summary["development_rows"] == 354
assert summary["validation_rows"] == 110
assert summary["matrix_guard"]["boundary_purge_rows_parsed_but_not_used"] == 6
assert summary["matrix_guard"]["locked_final_test_target_rows_parsed"] == 0
assert not summary["matrix_guard"]["final_test_target_values_available_to_step_11"]

## Six conservative candidates

Three bounded architectures—depth-1 stumps and two depth-2 variants—are each tested with full development history and a fixed recent five-year window. Every candidate uses CPU histogram trees, low learning rate, row and column subsampling, regularization, deterministic single-thread execution, and validation-MAE early stopping.

In [ ]:
candidates = artifacts["candidate_summary"]
display(candidates[[
    "validation_mae_rank", "candidate_id", "training_rows", "max_depth",
    "trees_used", "development_mae", "validation_mae", "validation_rmse",
    "validation_oos_r_squared", "validation_directional_accuracy",
]].round(4))
assert len(candidates) == 6
assert candidates["max_depth"].le(2).all()
assert candidates["stopped_early"].all()

## Selected prototype and fixed-baseline comparison

Validation MAE selects the prototype, with validation RMSE, lower complexity, and candidate ID as deterministic tie-breakers. This is a single-period prototype decision—not final model promotion.

In [ ]:
selected = pd.Series(summary["selected_candidate"], name="value")
display(selected.to_frame())
comparison = artifacts["comparison_metrics"]
display(comparison[[
    "strategy", "mae", "rmse", "oos_r_squared",
    "prediction_actual_correlation", "directional_accuracy",
    "prediction_minimum", "prediction_maximum",
]].round(4))
assert summary["comparison"]["xgboost_beats_ridge_mae"]
assert summary["comparison"]["xgboost_beats_ridge_rmse"]

## Overfitting and errors over time

The development metric is in-sample and is not compared as if it were out-of-sample evidence. The validation/development gap and yearly validation results expose generalization risk. Step 12 must repeat this through purged walk-forward folds.

In [ ]:
display(candidates[[
    "candidate_id", "development_mae", "validation_mae",
    "mae_generalization_gap", "mae_validation_to_development_ratio",
]].round(4))
yearly = artifacts["comparison_metrics_by_year"]
display(yearly.query("strategy in ['ridge', 'xgboost']")[[
    "calendar_year", "strategy", "sample_count", "mae", "rmse",
    "oos_r_squared", "directional_accuracy",
]].round(4))
worst = artifacts["residuals_over_time"].nlargest(10, "absolute_error")
display(worst[[
    "sample_date", "target_end_date", "actual", "prediction",
    "residual", "absolute_error", "vix_level_points",
]].round(4))

## VIX regimes and stress periods

VIX regimes use fixed, interpretable thresholds. COVID rows are necessarily an in-sample diagnostic because the frozen validation period begins in May 2021; 2022 tightening and 2023 regional-bank stress are out-of-sample validation diagnostics.

In [ ]:
display(artifacts["vix_regime_metrics"][[
    "vix_regime", "sample_count", "mae", "rmse",
    "oos_r_squared", "directional_accuracy",
]].round(4))
display(artifacts["crisis_period_metrics"][[
    "period", "evaluation_role", "sample_count", "mae", "rmse",
    "oos_r_squared", "directional_accuracy",
]].round(4))

## Feature importance and scenario-shock sensitivity

Gain importance describes how the fitted prototype split the selected training sample; it is not causality. Sensitivity holds the selected-window median state fixed, sets all eight scenario shocks to neutral zero, and varies one shock within its selected-training range.

In [ ]:
importance = artifacts["feature_importance"]
display(importance.head(15)[[
    "importance_rank", "feature", "normalized_total_gain",
    "split_count", "is_scenario_input",
]].round(4))
display(importance.query("is_scenario_input")[[
    "importance_rank", "feature", "normalized_total_gain", "split_count",
]].round(4))
display(artifacts["scenario_sensitivity_summary"].round(4))

## Step 11 decision

The selected XGBoost prototype improves this validation period, but only VIX Change among the eight scenario-shock features affects the selected stump model's median-state sensitivity sweep. Ridge and XGBoost therefore both advance as candidates to Step 12; neither is promoted from one validation period.

In [ ]:
checks = pd.Series(summary["validations"], name="passed")
display(checks.to_frame())
assert checks.all()
assert summary["target"] == "sp500"
assert summary["deterministic_repeat_max_prediction_difference"] == 0.0
assert summary["training_gate"] == "STEP_11_XGBOOST_PROTOTYPE_COMPLETE_READY_FOR_STEP_12_WALK_FORWARD_VALIDATION"
print(summary["prototype_decision"])
print(summary["training_gate"])